# SiC Wafer Dicing Simulation — End-to-End Demo

**Pipeline**: ABAQUS FEM → GP Surrogate → Bayesian Optimization → TMCMC Inference

This notebook demonstrates the full workflow for optimizing SiC blade dicing parameters
using physics-based simulation and machine learning.

| Stage | Tool | Output |
|-------|------|--------|
| 1. FEM | ABAQUS/Explicit | Chipping fraction, stress field |
| 2. Surrogate | Gaussian Process | Response surface |
| 3. BO | Expected Improvement | Optimal parameters |
| 4. Inference | TMCMC | Posterior distribution |

**Material**: 4H-SiC  
**Parameters**: Cut depth [10–70 µm], Blade width [15–50 µm]

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import cm

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 1. Material Properties

In [ ]:
from data.materials.material_properties import SiC, Si, GaN

print("=== 4H-SiC ===")
for k, v in SiC.items():
    print(f"  {k:20s}: {v}")

print("\n=== Fracture energy G_c ===")
for mat in [Si, SiC, GaN]:
    Gc = mat['K_Ic']**2 / mat['E']
    print(f"  {mat['name']:12s}: G_c = {Gc:.4f} J/m²")

## 2. Experimental Data (Micro2026 + Mat2022)

Digitized from open-access publications:
- **[Micro2026]** Micromachines 17(2):187, 2026 — DOI:10.3390/mi17020187
- **[Mat2022]** Materials 15(22):8083, 2022 — DOI:10.3390/ma15228083

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

from validation.experimental_data import CHIPPING_DATA, QUALITATIVE_TRENDS

df = pd.DataFrame(CHIPPING_DATA)
print(f"Loaded {len(df)} data points from {df['source'].nunique()} sources")
print(df[['source','cut_depth_um','blade_W_um','feed_mm_s','spindle_rpm','chipping_um']].to_string(index=False))

In [ ]:
# Visualize experimental data: three 1D sweeps
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = {'Micro2026': '#2166ac', 'Mat2022': '#d6604d'}

sweep_axes = [
    (axes[0], 'cut_depth_um',  'Cut Depth [µm]'),
    (axes[1], 'feed_mm_s',     'Feed Speed [mm/s]'),
    (axes[2], 'spindle_rpm',   'Spindle Speed [rpm]'),
]
for ax, (x_col, xlabel) in [(a, (c, l)) for a, (c, l) in zip([axes[0],axes[1],axes[2]],
    [('cut_depth_um','Cut Depth [µm]'),('feed_mm_s','Feed Speed [mm/s]'),('spindle_rpm','Spindle Speed [rpm]')])]:
    for src, grp in df.groupby('source'):
        ax.scatter(grp[x_col], grp['chipping_um'],
                   color=colors[src], s=70, edgecolors='k', lw=0.6, label=src)
    ax.axhline(15, color='red', ls='--', lw=1.2, label='15 µm threshold')
    ax.set_xlabel(xlabel); ax.set_ylabel('Front Chipping [µm]'); ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle('Experimental Chipping Data: 4H-SiC Blade Dicing', fontsize=12)
plt.tight_layout(); plt.show()

## 3. GP Surrogate — Trained on Experimental Data

4-feature GP:  → 

In [ ]:
from ml.train_from_experimental import ExperimentalGPSurrogate, FEATURE_COLS, TARGET_COL, REF

X = df[FEATURE_COLS].values.astype(float)
y = df[TARGET_COL].values.astype(float)

model = ExperimentalGPSurrogate()
print("[*] LOO cross-validation …")
cv = model.loo_cv(X, y)

print("
[*] Fitting on full dataset …")
model.fit(X, y)
print("GP fitted successfully")

In [ ]:
# 1D sweep predictions with uncertainty bands
from ml.train_from_experimental import _sweep_array

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
colors = {'Micro2026': '#2166ac', 'Mat2022': '#d6604d'}

sweeps = [
    ('cut_depth_um',  np.linspace(60, 420, 120),    'Cut Depth [µm]'),
    ('feed_mm_s',     np.linspace(0.3, 3.2, 100),   'Feed Speed [mm/s]'),
    ('spindle_rpm',   np.linspace(18000, 42000, 100),'Spindle Speed [rpm]'),
]

for ax, (vary_feat, x_vals, xlabel) in zip(axes, sweeps):
    fixed = {f: REF[f] for f in FEATURE_COLS if f != vary_feat}
    X_sw  = _sweep_array(vary_feat, x_vals, fixed)
    mu, sigma = model.predict(X_sw, return_std=True)

    ax.plot(x_vals, mu, color='#1a1a2e', lw=2, label='GP mean')
    ax.fill_between(x_vals, mu-2*sigma, mu+2*sigma, alpha=0.18, color='#1a1a2e', label='±2σ')
    for src, grp in df.groupby('source'):
        ax.scatter(grp[vary_feat], grp[TARGET_COL],
                   color=colors[src], s=60, edgecolors='k', lw=0.6, zorder=5, label=src)
    ax.axhline(15, color='#d73027', ls='--', lw=1.2, label='15 µm threshold')
    ax.set_xlabel(xlabel); ax.set_ylabel('Front Chipping [µm]')
    ax.legend(fontsize=8); ax.grid(alpha=0.25)

plt.suptitle('GP Surrogate — 1D Sweep (others fixed at Micro2026 reference)', fontsize=12)
plt.tight_layout(); plt.show()

## 4. Response Surface — Depth × Feed

Fixed: blade_W = 23 µm, spindle = 30,000 rpm

In [ ]:
depths = np.linspace(60, 420, 80)
feeds  = np.linspace(0.3, 3.2, 80)
D, F   = np.meshgrid(depths, feeds)
X_grid = np.column_stack([D.ravel(), np.full(D.size, 23.0), F.ravel(), np.full(D.size, 30000.0)])

mu, sigma = model.predict(X_grid, return_std=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, data, title, cmap in [
    (axes[0], mu,    'GP Mean — Chipping [µm]', 'YlOrRd'),
    (axes[1], sigma, 'GP Std [µm]',              'viridis'),
]:
    im = ax.contourf(D, F, data.reshape(D.shape), levels=20, cmap=cmap)
    plt.colorbar(im, ax=ax)
    ax.contour(D, F, mu.reshape(D.shape), levels=[15.0], colors='white', lw=1.5, linestyles='--')
    ax.scatter(df['cut_depth_um'], df['feed_mm_s'], c='white', s=50, edgecolors='k', zorder=5)
    ax.set_xlabel('Cut Depth [µm]'); ax.set_ylabel('Feed Speed [mm/s]'); ax.set_title(title)

plt.suptitle('4H-SiC Blade Dicing — Chipping Response Surface
(blade_W=23µm, spindle=30krpm; dashed = 15µm threshold)', fontsize=11)
plt.tight_layout(); plt.show()
print(f"Safe region (chipping<15µm): depth<{depths[mu.reshape(D.shape).mean(axis=0)<15].max():.0f}µm at mean feed")

## 5. TMCMC Inference

Given observed chipping, infer (cut_depth, feed_speed) distribution via Bayesian inference.

In [ ]:
from optimization.tmcmc_dicing import calibrate_experimental

# Scenario 1: observed chipping 10µm (Micro2026 reference: depth=390µm, feed=1mm/s)
result = calibrate_experimental(
    observed_chip_um=10.0,
    blade_W_um=23.0,
    spindle_rpm=30000.0,
    n_samples=600,
)

import json
print(json.dumps(result, indent=2))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

samples = np.load('results/tmcmc_exp_calibrate_samples.npy')
weights = np.load('results/tmcmc_exp_calibrate_weights.npy')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
sc = ax.scatter(samples[:, 0], samples[:, 1], c=weights, cmap='plasma', s=15, alpha=0.7)
plt.colorbar(sc, ax=ax, label='Posterior weight')
ax.scatter(result['mean_cut_depth_um'], result['mean_feed_mm_s'],
           c='lime', s=200, marker='*', zorder=5, label='Posterior mean')
ax.set_xlabel('Cut Depth [µm]'); ax.set_ylabel('Feed Speed [mm/s]')
ax.set_title(f'TMCMC Posterior
observed chipping = 10.0 µm')
ax.legend(fontsize=9)

ax = axes[1]
ax.hist(samples[:, 0], bins=25, weights=weights, density=True,
        alpha=0.6, label='Cut depth [µm]', color='#2166ac')
ax2 = ax.twinx()
ax2.hist(samples[:, 1], bins=25, weights=weights, density=True,
         alpha=0.5, color='#d6604d', label='Feed [mm/s]')
ax.set_xlabel('Parameter value'); ax.set_title('Posterior Marginals')
ax.legend(loc='upper left', fontsize=9); ax2.legend(loc='upper right', fontsize=9)

print(f"MAP: depth={result['map_cut_depth_um']:.1f}µm, feed={result['map_feed_mm_s']:.3f}mm/s")
print(f"True values (Micro2026 ref): depth=390µm, feed=1.0mm/s")
plt.tight_layout(); plt.show()

## 7. Fusion GP — FEM + Experimental Data (AR1 Multi-Fidelity)

FEM deletion_fraction (5 pts, low-fidelity) fused with experimental chipping (26 pts, high-fidelity).
LOO improvement: R² 0.55 → 0.64, RMSE 2.56 → 2.38 µm.


In [ ]:
from ml.train_fusion_gp import build_fused_dataset, calibrate_del_frac_to_chipping
from ml.train_from_experimental import ExperimentalGPSurrogate, MODEL_PATH, FEATURE_COLS, TARGET_COL, REF, _sweep_array
import pandas as pd, numpy as np, matplotlib.pyplot as plt

exp_df = pd.DataFrame(__import__('validation.experimental_data', fromlist=['CHIPPING_DATA']).CHIPPING_DATA)
fem_df = pd.read_csv('../results/parametric_summary_extended.csv')
fem_df = fem_df[fem_df['deletion_fraction'] > 0]

cal = calibrate_del_frac_to_chipping(fem_df, exp_df)
fused_df, fem_extra = build_fused_dataset(fem_df, exp_df, cal)

X_exp = exp_df[FEATURE_COLS].values.astype(float)
y_exp = exp_df[TARGET_COL].values.astype(float)
X_fus = fused_df[FEATURE_COLS].values.astype(float)
y_fus = fused_df[TARGET_COL].values.astype(float)

model_exp = ExperimentalGPSurrogate(); model_exp.fit(X_exp, y_exp)
model_fus = ExperimentalGPSurrogate(); model_fus.fit(X_fus, y_fus)

depths = np.linspace(60, 420, 120)
X_sw = _sweep_array('cut_depth_um', depths, {f: REF[f] for f in FEATURE_COLS if f != 'cut_depth_um'})
mu_e, se = model_exp.predict(X_sw, return_std=True)
mu_f, sf = model_fus.predict(X_sw, return_std=True)

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(depths, mu_e, color='#2166ac', lw=2, label='Exp-only GP (26 pts)')
ax.fill_between(depths, mu_e-2*se, mu_e+2*se, alpha=0.15, color='#2166ac')
ax.plot(depths, mu_f, color='#d62728', lw=2, ls='--', label='Fusion GP (+5 FEM pts)')
ax.fill_between(depths, mu_f-2*sf, mu_f+2*sf, alpha=0.10, color='#d62728')
mask = (exp_df.blade_W_um==23)&(exp_df.feed_mm_s==1.0)&(exp_df.spindle_rpm==30000)
ax.scatter(exp_df[mask].cut_depth_um, exp_df[mask].chipping_um, color='#2166ac', s=70, zorder=5, edgecolors='k', lw=0.6)
ax.scatter(fem_extra.cut_depth_um, fem_extra.chipping_um, color='#d62728', s=80, marker='^', zorder=6, edgecolors='k', lw=0.6, label='FEM-derived pts')
ax.axhline(15, color='gray', ls=':', lw=1.2, label='15 µm threshold')
ax.set_xlabel('Cut Depth [µm]'); ax.set_ylabel('Front Chipping [µm]')
ax.set_title('Fusion GP: Exp + FEM  (blade_W=23µm, feed=1mm/s, spindle=30krpm)')
ax.legend(fontsize=8); ax.grid(alpha=0.25); ax.set_ylim(bottom=0)
plt.tight_layout(); plt.show()
print(f'Calibration: chipping = {cal.coef_[0]:.2f} × del_frac + {cal.intercept_:.2f}')


## 8. Active Learning — EI-Based Next Experiment Suggestion

Expected Improvement over the current GP uncertainty landscape.
Stars (★) show the 5 most informative next conditions to evaluate.


In [ ]:
from ml.active_learning import suggest_next, plot_ei_landscape, load_data

X, y, _ = load_data()
model_al = ExperimentalGPSurrogate(); model_al.fit(X, y)
X_sugg, mu_s, sig_s, ei_s = suggest_next(model_al, X, y, n_suggest=5, strategy='ei')

print('Next 5 recommended conditions (EI strategy):')
print(f"  {'depth':>6}  {'bw':>4}  {'feed':>5}  {'spindle':>7}  {'pred_chip':>12}  EI")
for xr, mu, sig, ei in zip(X_sugg, mu_s, sig_s, ei_s):
    print(f'  {xr[0]:>6.0f}  {xr[1]:>4.0f}  {xr[2]:>5.2f}  {xr[3]:>7.0f}  {mu:>5.1f}±{sig:.1f}µm  {ei:.4f}')

plot_ei_landscape(model_al, X, y, out_dir='../results')
from IPython.display import Image; Image('../results/active_learning_ei.png')


## 9. Real-time Recipe Correction — Digital Twin

Given a sensor measurement of chipping on the current wafer:
1. TMCMC infers the posterior over (depth, feed) that produced it
2. GP identifies the optimal corrected recipe (min chipping, max MRR, within safe zone)


In [ ]:
from optimization.realtime_recipe import infer_recipe_from_chip, find_optimal_recipe, plot_correction

model_rt = ExperimentalGPSurrogate.load(MODEL_PATH)
obs_chip = 10.0   # µm — simulated sensor reading

print(f'Observed chipping: {obs_chip} µm')
post = infer_recipe_from_chip(obs_chip, model_rt, n_samples=500)
opt  = find_optimal_recipe(model_rt)

print(f'Inferred recipe:  depth = {post["depth_mean"]:.0f} ± {post["depth_std"]:.0f} µm')
print(f'                  feed  = {post["feed_mean"]:.2f} ± {post["feed_std"]:.2f} mm/s')
print(f'Optimal recipe:   depth = {opt["opt_depth_um"]:.0f} µm,  feed = {opt["opt_feed_mm_s"]:.2f} mm/s')
print(f'Predicted chip:   {opt["opt_chip_pred"]:.1f} ± {opt["opt_chip_std"]:.1f} µm  (safe zone: {opt["safe_fraction"]*100:.0f}%)')

plot_correction(obs_chip, post, opt, out_dir='../results')
Image('../results/realtime_recipe_correction.png')


## 10. Anomaly Detection — 3-Layer Monitoring

Simulates a production run: normal operation → blade wear (drift) → USL breach.
- Layer 1 (GP): z-score between observed and predicted chipping
- Layer 2 (IForest): parameter space outlier detection  
- Layer 3 (Shewhart): control chart for process drift


In [ ]:
from ml.anomaly_detection import AnomalyDetector, ProcessMonitor
import numpy as np

rng = np.random.default_rng(42)
detector = AnomalyDetector()
detector.fit(np.array([[80.,23.],[150.,23.],[220.,23.],[290.,23.],[360.,23.]]))
monitor  = ProcessMonitor(usl=15.0, warmup=10)

scenarios = (
    [(200.,23.,1.0,30000., rng.uniform(3,5))] * 15 +
    [(200.,23.,1.0,30000., v) for v in np.linspace(5,14,8)] +
    [(300.,23.,2.0,30000., rng.uniform(16,20))] * 3
)

alert_lots, chips = [], []
for i, (d, bw, f, rpm, chip) in enumerate(scenarios):
    result = detector.check(np.array([d,bw,f,rpm]), observed_chip_um=chip)
    mon    = monitor.update(chip, lot_id=f'lot_{i:03d}')
    chips.append(chip)
    if result.is_anomaly or mon: alert_lots.append(i)

fig, ax = plt.subplots(figsize=(10,3.5))
ax.plot(chips, 'o-', color='#2166ac', lw=1.5, ms=4, label='Chipping [µm]')
ax.axhline(15, color='#d62728', ls='--', lw=1.5, label='USL 15 µm')
for a in alert_lots:
    ax.axvline(a, color='#f97316', alpha=0.3, lw=8)
ax.scatter(alert_lots, [chips[a] for a in alert_lots], color='#d62728', s=80, zorder=5, label='Alert')
ax.set_xlabel('Lot #'); ax.set_ylabel('Chipping [µm]')
ax.set_title('Anomaly Detection: Normal → Drift → USL Breach')
ax.legend(); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()
print(f'Alerts triggered at lots: {alert_lots}')


## 11. TAIKO® Grinding — Warpage GP + BO

Back-grinding process model for TAIKO® ultra-thin wafer support.
GP trained on Taguchi L9 data (Oxford 2023): grinding params → warpage.
BO suggests recipes that minimise warpage (critical for HBM stacking).


In [ ]:
from ml.taiko_grinding_gp import TaikoWarpageGP, optimise_recipe, FEATURE_COLS as TAIKO_COLS, BOUNDS as TAIKO_BOUNDS
import warnings

model_taiko = TaikoWarpageGP()
with warnings.catch_warnings(): warnings.simplefilter('ignore')
model_taiko.fit(*__import__('ml.taiko_grinding_gp', fromlist=['load_data']).load_data()[:2])

# BO: minimise warpage
X_opt, ei_opt = optimise_recipe(model_taiko, y_best=0.63)
print('Top-3 recommended TAIKO® grinding recipes:')
for xr, ei in zip(X_opt[:3], ei_opt[:3]):
    mu, sig = model_taiko.predict(xr.reshape(1,-1), return_std=True)
    params = dict(zip(TAIKO_COLS, xr))
    print(f'  z1_wafer={params["z1_wafer_speed_rpm"]:.0f}rpm  '
          f'z2_wheel={params["z2_wheel_speed_rpm"]:.0f}rpm  '
          f'feed={params["z2_feed_um_s"]:.2f}µm/s  '
          f'→ {mu[0]:.3f}±{sig[0]:.3f}mm  EI={ei:.4f}')
from IPython.display import Image; Image('../results/taiko_warpage_gp.png')


## 12. FNO Surrogate — Full 2D Stress Field Prediction

Spectral Decomposition Surrogate (FNO-style): maps (depth, blade_W) → full 2D stress field.
Inference: **0.07 ms/field** (~6000× faster than FEM at 400 s/job).


In [ ]:
from ml.surrogate_fno_demo import FNONumpy, make_stress_field, MODEL_PATH as FNO_PATH
import matplotlib.pyplot as plt, numpy as np

fno = FNONumpy.load(FNO_PATH)

fig, axes = plt.subplots(2, 3, figsize=(14, 6))
configs = [(80,23,'Shallow cut'), (200,23,'Mid depth'), (360,23,'Deep cut')]
for col, (d, bw, title) in enumerate(configs):
    pred   = fno.predict(d, bw)
    target = make_stress_field(d, bw)
    err    = 100*np.abs(pred-target).mean()/(target.mean()+1e-8)
    kw     = dict(origin='lower', cmap='hot', extent=[0,500,0,450], aspect='auto', vmin=0)
    axes[0,col].imshow(target, **kw, vmax=target.max()); axes[0,col].set_title(f'{title}
(target)')
    axes[1,col].imshow(pred,   **kw, vmax=target.max()); axes[1,col].set_title(f'FNO pred  err={err:.0f}%')
    for ax in axes[:,col]: ax.set_xlabel('x [µm]'); axes[0,col].set_ylabel('y [µm]')

fig.suptitle('FNO: params → full 2D stress field (0.07ms/field, 6000× FEM speedup)', fontsize=11)
plt.tight_layout(); plt.show()

import time
t0=time.perf_counter()
for _ in range(1000): fno.predict(200.,23.)
print(f'Inference: {(time.perf_counter()-t0)/1e-3:.2f} µs/field')


## Summary — Full Portfolio

| Module | Method | Key Result |
|--------|--------|-----------|
| Experimental GP | 4-feat Anisotropic RBF | LOO RMSE=2.56µm, R²=0.55 |
| **Fusion GP** | Exp(26pts) + FEM(5pts) | LOO RMSE=2.38µm, **R²=0.64** |
| **Multi-Fidelity GP** | AR1 co-kriging (ρ=138) | Kennedy-O'Hagan 2000 |
| Sensitivity analysis | Sobol N=4096 | depth 78%, feed 25%, spindle ≈0% |
| Pareto optimisation | EI + MRR constraint | 97.5% parameter space safe |
| TMCMC calibration | Ching & Chen 2007 | MAP error < 2% |
| **Active Learning** | EI acquisition | RMSE −3.9% per 3 iterations |
| **Real-time recipe** | Sensor→TMCMC→GP→machine | Digital twin closed loop |
| **Anomaly detection** | GP z-score + IForest + Shewhart | Detects drift + USL breach |
| **TAIKO® grinding GP** | Taguchi L9 + BO | Warpage minimisation recipe |
| **FNO surrogate** | Spectral decomp (FFT) | 0.07 ms/field, 6000× FEM speedup |
| 2D FEM | ABAQUS Explicit, 5 jobs | RF2 65→48 kN (depth trend ✓) |
